# slv_controles_integridad
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Silver (nivel 2) · controles de integridad entre entidades |
| Propósito | - control_totales_pedido: por pedido, suma de ítems frente al total de cabecera, pedidos sin ítems y cliente inexistente. - control_integridad: filas huérfanas por relación (ítems sin pedido, pagos sin pedido, pedidos sin cliente, etc.). |
| Por qué | la integridad entre Orders, OrderItems y Payments se mide, no se supone. Con FK reales en la fuente, un huérfano solo aparece por orden de llegada. |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 2.0 · 2026-10-12 |

## 1. Librerías y módulos comunes
- `dlt`: API de los pipelines declarativos (vistas, tablas, AUTO CDC, expectations).
- `comun` / `comun_gold` y `reglas_calidad`: utilidades y reglas compartidas por todas las entidades (archivos `.py` de la misma carpeta; el pipeline los encuentra gracias a `root_path`).

In [ ]:
import dlt
from pyspark.sql import functions as F

## 2. Tabla `control_totales_pedido`
Integridad cabecera-detalle por pedido: suma de ítems frente al total, pedidos sin ítems 

In [ ]:
@dlt.table(name="control_totales_pedido",
           comment="Integridad cabecera-detalle por pedido: suma de ítems frente al total, pedidos sin ítems "
                   "y cliente inexistente.")
def control_totales_pedido():
    pedidos = dlt.read("pedidos")
    items = (dlt.read("items_pedido").groupBy("id_pedido")
             .agg(F.count("*").alias("n_items"), F.sum("monto_item").alias("suma_items")))
    clientes = dlt.read("clientes").select("id_cliente", F.lit(True).alias("_cliente_existe"))
    return (pedidos.join(items, "id_pedido", "left").join(clientes, "id_cliente", "left")
            .select("id_pedido", "id_cliente", "fecha_pedido", "estado", "monto_total",
                    F.coalesce("n_items", F.lit(0)).alias("n_items"),
                    F.coalesce("suma_items", F.lit(0)).cast("decimal(14,2)").alias("suma_items"),
                    (F.coalesce("n_items", F.lit(0)) == 0).alias("es_sin_items"),
                    (F.abs(F.col("monto_total") - F.coalesce("suma_items", F.lit(0))) > 0.01)
                    .alias("es_total_descuadrado"),
                    F.col("_cliente_existe").isNull().alias("es_cliente_inexistente")))

## 3. Tabla `control_integridad`
Resumen de integridad referencial entre entidades de silver (filas huérfanas por relación).

In [ ]:
@dlt.table(name="control_integridad",
           comment="Resumen de integridad referencial entre entidades de silver (filas huérfanas por relación).")
def control_integridad():
    def huerfanos(hija, col, padre, col_padre, relacion):
        h = dlt.read(hija).select(F.col(col).alias("k")).filter("k IS NOT NULL")
        p = dlt.read(padre).select(F.col(col_padre).alias("k")).distinct()
        return (h.join(p, "k", "left_anti").agg(F.count("*").alias("filas_huerfanas"))
                 .select(F.lit(relacion).alias("relacion"), "filas_huerfanas"))
    return (huerfanos("items_pedido", "id_pedido", "pedidos", "id_pedido", "items_pedido -> pedidos")
            .unionByName(huerfanos("items_pedido", "id_producto", "productos", "id_producto", "items_pedido -> productos"))
            .unionByName(huerfanos("pagos", "id_pedido", "pedidos", "id_pedido", "pagos -> pedidos"))
            .unionByName(huerfanos("pedidos", "id_cliente", "clientes", "id_cliente", "pedidos -> clientes"))
            .unionByName(huerfanos("tickets", "id_cliente", "clientes", "id_cliente", "tickets -> clientes")))